In [ ]:
!pip install --upgrade gensim

In [ ]:
import pandas as pd
import zipfile
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from sklearn.preprocessing import LabelEncoder
from gensim.models import Word2Vec
from tensorflow.keras.preprocessing.text import Tokenizer
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report
from tensorflow.keras.callbacks import EarlyStopping
import re


In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle (2).json


{'kaggle (2).json': b'{"username":"sahnoon","key":"005b73da67dc2954e68363a872638520"}'}

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
!kaggle datasets download -d adhamelkomy/bank-customer-complaint-analysis

Dataset URL: https://www.kaggle.com/datasets/adhamelkomy/bank-customer-complaint-analysis
License(s): CC0-1.0
bank-customer-complaint-analysis.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
with zipfile.ZipFile("/content/bank-customer-complaint-analysis.zip", 'r') as zip_ref:
    zip_ref.extractall()

In [ ]:
orginal_df = pd.read_csv("/content/complaints.csv")

In [ ]:
df = orginal_df.copy()
df.head()

,Unnamed: 0,product,narrative
0,0,credit_card,purchase order day shipping amount receive pro...
1,1,credit_card,forwarded message date tue subject please inve...
2,2,retail_banking,forwarded message cc sent friday pdt subject f...
3,3,credit_reporting,payment history missing credit report speciali...
4,4,credit_reporting,payment history missing credit report made mis...


In [ ]:
df.drop(["Unnamed: 0"], axis=1, inplace=True)

In [ ]:
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

In [ ]:
df.shape

(124676, 2)

In [ ]:
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def text_preprocessing(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove special chars and digits
    words = word_tokenize(text)
    pos_tags = pos_tag(words)
    lemmatized_words = []
    for word, tag in pos_tags:
        if len(word) > 2 and word.isalpha() and word not in stop_words:
            pos = get_wordnet_pos(tag)
            lemma = lemmatizer.lemmatize(word, pos)
            lemmatized_words.append(lemma)
    return ' '.join(lemmatized_words).strip()


In [ ]:
df['cleaned_text'] = df['narrative'].apply(text_preprocessing)

In [ ]:
df[['narrative', 'cleaned_text']].head()

,narrative,cleaned_text
0,purchase order day shipping amount receive pro...,purchase order day ship amount receive product...
1,forwarded message date tue subject please inve...,forward message date tue subject please invest...
2,forwarded message cc sent friday pdt subject f...,forward message send friday pdt subject final ...
3,payment history missing credit report speciali...,payment history miss credit report specialize ...
4,payment history missing credit report made mis...,payment history miss credit report make mistak...


In [ ]:
label_encoder = LabelEncoder()
df['product_encoded'] = label_encoder.fit_transform(df['product'])

In [ ]:
dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

{'credit_card': 0,
 'credit_reporting': 1,
 'debt_collection': 2,
 'mortgages_and_loans': 3,
 'retail_banking': 4}

In [ ]:
x = df['cleaned_text']
y = df['product_encoded']

In [ ]:
tokenized_sentences = [sentence.split() for sentence in x]

In [ ]:
w2v_model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=100,
    window=5,
    min_count=2,
    sg=1,
    workers=4
)




In [ ]:
w2v_model.wv.most_similar("credit", topn=5)


[('report', 0.7474378943443298),
 ('creidt', 0.6934356093406677),
 ('issac', 0.6881518363952637),
 ('crefit', 0.6801400184631348),
 ('debut', 0.6688819527626038)]

In [ ]:


tokenizer = Tokenizer()
tokenizer.fit_on_texts(x)
word_index = tokenizer.word_index


In [ ]:
sequences = tokenizer.texts_to_sequences(x)


In [ ]:


max_len = max(len(seq) for seq in sequences)
X_padded = pad_sequences(sequences, maxlen=max_len, padding='post')


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X_padded, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:


class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))


In [ ]:

embedding_dim = 100  # Must match your Word2Vec vector size
embedding_matrix = np.zeros((len(word_index) + 1, embedding_dim))

for word, i in word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]


In [ ]:


model = Sequential()

# Embedding layer with pretrained Word2Vec
model.add(Embedding(
    input_dim=len(word_index) + 1,
    output_dim=embedding_dim,
    weights=[embedding_matrix],
    input_length=max_len,
    trainable=True  # or True if you want to fine-tune Word2Vec
))

# BiLSTM layer to learn word sequences
model.add(Bidirectional(LSTM(128, return_sequences=False)))
model.add(Dropout(0.5))

# Fully connected layer
model.add(Dense(64, activation='relu'))

# Final output layer (5 classes = 5 neurons)
model.add(Dense(5, activation='softmax'))

# Compile the model
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

model.summary()


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │     4,323,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,323,800 (16.49 MB)

 Trainable params: 4,323,800 (16.49 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:


early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.2,
    class_weight=class_weights,
    callbacks=[early_stop]  # ⬅️ this will stop training early if val_loss stops improving
)


Epoch 1/15
1247/1247 ━━━━━━━━━━━━━━━━━━━━ 587s 468ms/step - accuracy: 0.7461 - loss: 0.6903 - val_accuracy: 0.8209 - val_loss: 0.4949
Epoch 2/15
 144/1247 ━━━━━━━━━━━━━━━━━━━━ 7:22 401ms/step - accuracy: 0.8182 - loss: 0.4860

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)


780/780 ━━━━━━━━━━━━━━━━━━━━ 52s 67ms/step - accuracy: 0.8400 - loss: 0.4513
Test Accuracy: 0.8416345715522766


In [ ]:


y_pred = model.predict(X_test)
y_pred_classes = y_pred.argmax(axis=1)

print(classification_report(y_test, y_pred_classes))


780/780 ━━━━━━━━━━━━━━━━━━━━ 50s 64ms/step
              precision    recall  f1-score   support

           0       0.77      0.79      0.78      3005
           1       0.92      0.84      0.88     11261
           2       0.76      0.82      0.79      4223
           3       0.78      0.88      0.83      3752
           4       0.85      0.88      0.87      2695

    accuracy                           0.84     24936
   macro avg       0.82      0.84      0.83     24936
weighted avg       0.85      0.84      0.84     24936

